In [79]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [80]:
data = pd.read_csv('IMDB Dataset.csv')
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [81]:
data['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [82]:
df = data.sample(10000)

In [83]:
df.shape

(10000, 2)

In [84]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 14038 to 25029
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     10000 non-null  object
 1   sentiment  10000 non-null  object
dtypes: object(2)
memory usage: 234.4+ KB


In [85]:
df['sentiment'].replace({'positive':1,'negative':0},inplace=True)

/tmp/ipython-input-4240984564.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['sentiment'].replace({'positive':1,'negative':0},inplace=True)
/tmp/ipython-input-4240984564.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'].replace({'positive':1,'negative':0},inplace=True)


In [86]:
df.head()

,review,sentiment
14038,2/3 of this movie is recycled footage of the p...,0
47664,"Dirty Sanchez is the more extreme, British ver...",1
8195,"Paul Hennessy and his wife, Cate must deal wit...",1
19490,I purchased the BLOOD CASTLE DVD on eBay for a...,0
2180,"This movie was made by Daiei Studios, known fo...",1


In [87]:
# removing html tags

import re
def remove_tags(raw_text):
    cleaned_text = re.sub(re.compile('<.*?>'), '', raw_text)
    return cleaned_text

In [88]:
df['review'] = df['review'].apply(remove_tags)

In [91]:
df['review'][47664]

"Dirty Sanchez is the more extreme, British version of Jackass in which the four boys (Pritchard, Dainton, Joycey and Pancho) go to great lengths to hurt and humiliate each other. The reason this show is better than Jackass is because most of the stunts are not planned which makes the reaction much more funny. There are 3 series of the show, the first follows them around and takes a long look at their lives eg. there's an episode on their love lives,jobs etc. The second series sends the boys to try out different occupations. The third follows their European tour. It seems that the boys get more and more daring as the show progresses through the series. In my opinion the third series is the best, but trust me when i say, if you have a week stomach DO NOT WATCH, as you are lightly to see a fair amount of blood and puke in every episode."

In [92]:
print(df.index)

Index([14038, 47664,  8195, 19490,  2180, 26108, 21256, 10387, 24253, 18591,
       ...
       26598, 28925, 16478, 45019, 22517, 44242,  8192,  6582, 15776, 25029],
      dtype='int64', length=10000)


In [93]:
# converting to lower

def lower_text(text):
    return text.lower()

In [94]:
df['review'] = df['review'].apply(lower_text)

In [96]:
df['review'][47664]

"dirty sanchez is the more extreme, british version of jackass in which the four boys (pritchard, dainton, joycey and pancho) go to great lengths to hurt and humiliate each other. the reason this show is better than jackass is because most of the stunts are not planned which makes the reaction much more funny. there are 3 series of the show, the first follows them around and takes a long look at their lives eg. there's an episode on their love lives,jobs etc. the second series sends the boys to try out different occupations. the third follows their european tour. it seems that the boys get more and more daring as the show progresses through the series. in my opinion the third series is the best, but trust me when i say, if you have a week stomach do not watch, as you are lightly to see a fair amount of blood and puke in every episode."

In [97]:
# removing special characters

def remove_special_character(text):
  x = ''

  for i in text:
    if i.isalnum():
      x = x + i
    else:
      x = x + ' '

  return x

In [98]:
df['review'] = df['review'].apply(remove_special_character)

In [99]:
df['review'][47664]

'dirty sanchez is the more extreme  british version of jackass in which the four boys  pritchard  dainton  joycey and pancho  go to great lengths to hurt and humiliate each other  the reason this show is better than jackass is because most of the stunts are not planned which makes the reaction much more funny  there are 3 series of the show  the first follows them around and takes a long look at their lives eg  there s an episode on their love lives jobs etc  the second series sends the boys to try out different occupations  the third follows their european tour  it seems that the boys get more and more daring as the show progresses through the series  in my opinion the third series is the best  but trust me when i say  if you have a week stomach do not watch  as you are lightly to see a fair amount of blood and puke in every episode '

In [100]:
#  remove stop words

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [101]:
stop_words = stopwords.words('english')
stop_words

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [102]:
def remove_stopwords(text):
  x = []

  for i in text.split():
    if i not in stop_words:
      x.append(i)

  y = x[:]
  x.clear()
  return y

In [103]:
df['review'] = df['review'].apply(remove_stopwords)

In [104]:
df['review'][47664]

['dirty',
 'sanchez',
 'extreme',
 'british',
 'version',
 'jackass',
 'four',
 'boys',
 'pritchard',
 'dainton',
 'joycey',
 'pancho',
 'go',
 'great',
 'lengths',
 'hurt',
 'humiliate',
 'reason',
 'show',
 'better',
 'jackass',
 'stunts',
 'planned',
 'makes',
 'reaction',
 'much',
 'funny',
 '3',
 'series',
 'show',
 'first',
 'follows',
 'around',
 'takes',
 'long',
 'look',
 'lives',
 'eg',
 'episode',
 'love',
 'lives',
 'jobs',
 'etc',
 'second',
 'series',
 'sends',
 'boys',
 'try',
 'different',
 'occupations',
 'third',
 'follows',
 'european',
 'tour',
 'seems',
 'boys',
 'get',
 'daring',
 'show',
 'progresses',
 'series',
 'opinion',
 'third',
 'series',
 'best',
 'trust',
 'say',
 'week',
 'stomach',
 'watch',
 'lightly',
 'see',
 'fair',
 'amount',
 'blood',
 'puke',
 'every',
 'episode']

In [105]:
df.head()

,review,sentiment
14038,"[2, 3, movie, recycled, footage, previous, mov...",0
47664,"[dirty, sanchez, extreme, british, version, ja...",1
8195,"[paul, hennessy, wife, cate, must, deal, two, ...",1
19490,"[purchased, blood, castle, dvd, ebay, bucks, k...",0
2180,"[movie, made, daiei, studios, known, gamera, m...",1


In [107]:
#  stemming

from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [109]:
a = []

def stemming(text):
  for i in text:
      a.append(ps.stem(i))

  b = a[:]
  a.clear()
  return b

In [110]:
df['review'] = df['review'].apply(stemming)

In [111]:
df.head()

,review,sentiment
14038,"[2, 3, movi, recycl, footag, previou, movi, fa...",0
47664,"[dirti, sanchez, extrem, british, version, jac...",1
8195,"[paul, hennessi, wife, cate, must, deal, two, ...",1
19490,"[purchas, blood, castl, dvd, ebay, buck, know,...",0
2180,"[movi, made, daiei, studio, known, gamera, mov...",1


In [112]:
#  join back string

def join_back(list_input):
  return " ".join(list_input)

In [113]:
df['review'] = df['review'].apply(join_back)

In [114]:
df.head()

,review,sentiment
14038,2 3 movi recycl footag previou movi fact sadli...,0
47664,dirti sanchez extrem british version jackass f...,1
8195,paul hennessi wife cate must deal two teenag d...,1
19490,purchas blood castl dvd ebay buck know say dis...,0
2180,movi made daiei studio known gamera movi samur...,1


In [127]:
#  now our df is read and final task is to convert this data into vectors using CountVectrorizer here but there are several ways to do it
# colomns - word and their count and rows - reviews
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()

In [133]:
X = cv.fit_transform(df['review']).toarray()

In [134]:
X.shape

(10000, 36280)

In [135]:
y = df.iloc[:,-1].values

In [136]:
y.shape

(10000,)

In [137]:
y

array([0, 1, 1, ..., 1, 1, 0])

In [138]:
# Now we have our X and y in numbers and ready to pass to model.

# split data in training and testing

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=123)

In [139]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((8000, 36280), (2000, 36280), (8000,), (2000,))

In [140]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [141]:
nbg = GaussianNB()
nbm = MultinomialNB()
nbb = BernoulliNB()

In [142]:
nbg.fit(X_train, y_train)
nbm.fit(X_train, y_train)
nbb.fit(X_train, y_train)

BernoulliNB()

In [143]:
y_pred_g = nbg.predict(X_test)
y_pred_m = nbm.predict(X_test)
y_pred_b = nbb.predict(X_test)

In [145]:
print("Gaussian: ", accuracy_score(y_test, y_pred_g))
print("Multinomial: ", accuracy_score(y_test, y_pred_m))
print("Bernoulli: ", accuracy_score(y_test, y_pred_b))

Gaussian:  0.6265
Multinomial:  0.8545
Bernoulli:  0.842
